# ResNet50 Transfer LEarning for Urban Street Classification
https://www.kaggle.com/datasets/erickendric/tree-dataset-of-urban-street-classification-tree

This notebook investigates transfer learning with ResNEt50 for the Tree Dataset of Urban Street Classification.

The experiment follows four stages:
1. Inspect and preprocess the dataset
2. Train and validate an ImageNet-pretrained ResNEt50.
3. Evaluate the model on the independent test set.
4. Analyse classification erors and investigate possible data-quality imrpovements

The preprocessing decisions are based on observations from the dataset rather than arbitrary image modifications.



### Dataset

The dataset contains 4,804 images divided into training, validation, and test sets.

| Split | Images |
|---|---:|
| Train | 3,850 |
| Validation | 482 |
| Test | 472 |
| Total | 4,804 |

## Environment and imports

These imports provide the tools needed for filesystem inspection, counting, hashing, image analysis, visualization, and Pytorch data preparation.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import random
import hashlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as display
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder

## 1. Dataset inspection

**1.1 Verify the dataset path and folder structure:**  to assure that the dataset path is accessbile and prints its directory structure. 



In [ ]:
dataset_path = Path("D:\\NEUTRAL_TOMMI\\GROUPPROJECT\\tree")
print(dataset_path.name + "/")

for path in sorted(dataset_path.iterdir()):
    if path.is_dir():
        print(f"   {path.name}/")       

**1.2 Count images in each folder**: to identify how many images within each split and visualize its proportions in dataset. The result show that there are 3,850 training images, 482 validation images, and 472 test images - 4,804 images in total

In [ ]:
# Count the number of images in each split (train, val, test) and return a dictionary with the counts

def count_images_in_splits(dataset_path, splits=("train", "val", "test")):
    
    image_extensions = {
        ".jpg", ".jpeg", ".png", ".gif",
        ".bmp", ".tiff", ".webp"
    }
    # Set to store the counts of images in each split
    
    image_counts = {}

    for split in splits:
        split_path = Path(dataset_path) / split

        image_count = sum(
            1
            for file in split_path.rglob("*")
            if file.is_file()
            and file.suffix.lower() in image_extensions
        )

        image_counts[split] = image_count

    return image_counts

In [ ]:
image_counts = count_images_in_splits(dataset_path)
print(f"Total number of images: {image_counts}")

**1.3 Identify missing classes between splits**

This is the direct consistency check behind the previous visualization. The exploration shows that all three splits contain 23 classes, and no class is missing from train, validation, or test.

In [ ]:
# Count the number of classes in each split (train, val, test) and return a dictionary with the counts

train_class = {
    folder.name for folder in (dataset_path / "train").iterdir() if folder.is_dir()
}

val_class = {
    folder.name for folder in (dataset_path / "val").iterdir() if folder.is_dir()
}

test_class = {
    folder.name for folder in (dataset_path / "test").iterdir() if folder.is_dir()
}

print("Train classes:", len(train_class))
print("Validation classes:", len(val_class))
print("Test classes:", len(test_class))

print("\nClasses missing from validation set:", sorted(train_class - val_class))
print("Classes missing from test set:", sorted(train_class - test_class))
print("Classes missing from training set:", sorted(val_class - train_class))

Find duplicate images within splits and between splits

In [ ]:
# Check for duplicates within each split using MD5 hashing

def md5_hash(file_path, chunk_size=1024 * 1024):
    image_extensions
    hash_md5 = hashlib.md5()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

# Store duplicate information in a dictionary
duplicate_info = {}

for split in ["train", "val", "test"]:
    split_path = dataset_path / split

    print(f"\n{'='*60}")
    print(f"\nChecking for duplicates in {split} set.")
    print(f"{'='*60}")

    hash_to_files = defaultdict(list)
    image_files =[
        file for file in split_path.rglob("*") 
        if file.is_file() and file.suffix.lower() in image_extensions
    ]

    print(f"Image files found in {split} set: {len(image_files)}")

    for image_file in image_files:
        try:
            file_hash = md5_hash(image_file)
            hash_to_files[file_hash].append(image_file)
        except Exception as e:
            print(f"Error processing {image_file}: {e}")

    # Find duplicate groups (hashes with more than one file)
    duplicates = {file_hash: files for file_hash, files in hash_to_files.items() if len(files) > 1}
    # Count duplicate files
    duplicate_count = sum(len(files) - 1 for files in duplicates.values())
    duplicate_info[split] = {
        "duplicate_groups": len(duplicates),
        "duplicate_files": duplicate_count
    }
    print(f"Duplicate groups found in {split} set: {len(duplicates)}")
    print(f"Total duplicate files in {split} set: {duplicate_count}")

    if not duplicates:
        print(f"No duplicates found in {split} set.")

In [ ]:
# Check for cross-split duplicates between splits using MD5 hashing

def find_cross_split_duplicates(dataset_path,split1, split2, image_extensions):
    hash_to_files = defaultdict(list)
    cross_split_duplicates = defaultdict(set)
    # Create a dictionary to store the MD5 hash of each image and the split it belongs to
    for split in [split1, split2]:
        split_path = dataset_path / split
        image_files = [
            file for file in split_path.rglob("*") 
            if file.is_file() and file.suffix.lower() in image_extensions
        ]

        for image_file in image_files:
            try:
                file_hash = md5_hash(image_file)
                hash_to_files[file_hash].append((split, image_file))
            except Exception as e:
                print(f"Error processing {image_file}: {e}")

    for file_hash, files in hash_to_files.items():
        if len(files) > 1:
            splits_present = {split for split, _ in files}
            if len(splits_present) > 1:
                cross_split_duplicates[tuple(sorted(splits_present))].add(file_hash)

    return cross_split_duplicates
print("\nChecking for cross-split duplicates...")
cross_split_duplicates = find_cross_split_duplicates(dataset_path, "train", "val")

if cross_split_duplicates:
    print("Found cross-split duplicates:")
    for splits, hashes in cross_split_duplicates.items():
        print(f"  {splits}: {len(hashes)} duplicates")
else:
    print("No cross-split duplicates found.")

**1.4 Visual dataset inspection**

Visually inspect one random image from every training class

This grid provides a quick qualitative check of the species labels and image content.
With the purpose of looking for incorrect labels, unusual images, empty/corrupted-looking samples, inconsistent backgrounds, or image where the tree is barely visible.

In [ ]:
# Visualize random sample images from each class in the training set

import math

image_extensions
train_path = dataset_path / "train"
classes = sorted(
    folder.name for folder in train_path.iterdir() if folder.is_dir()
)

n_cols = 4
n_rows = math.ceil(len(classes) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = np.asarray(axes).flatten()
# Plot a random image from each class
for ax_index, class_name in enumerate(classes):
    class_path = train_path / class_name
    image_files = [
        file for file in class_path.rglob("*")
        if file.is_file() and file.suffix.lower() in image_extensions
    ]

    if image_files:
        random_image_file = random.choice(image_files)
        image = Image.open(random_image_file)
        axes[ax_index].imshow(image)
        axes[ax_index].set_title(class_name, fontsize=10, pad=8)
        axes[ax_index].axis("off")

for ax_index in range(len(classes), len(axes)):
    axes[ax_index].axis("off")

fig.suptitle("Random Sample Images from Each Class in Training Set", fontsize=16)
plt.subplots_adjust(wspace=0.4, hspace=0.6)
plt.show()

**Compare one random example of the same species across splits**

A random selected species is displayed once from train, validation, and test. This gives a quick visual check that the same class has broadly comparable image content across splits. 

In [ ]:
# Visualize random images from train, validation, and test sets for a randomly selected species

species = random.choice(list(train_class))

train_species_path = dataset_path / "train" / species
val_species_path = dataset_path / "val" / species
test_species_path = dataset_path / "test" / species

image_extensions

def get_random_image_from_species(folder):
    return [
        file for file in folder.rglob("*") 
        if file.is_file() and file.suffix.lower() in image_extensions
    ]    

train_species_images = get_random_image_from_species(train_species_path)
val_species_images = get_random_image_from_species(val_species_path)
test_species_images = get_random_image_from_species(test_species_path)

random_train_image = random.choice(train_species_images)
random_val_image = random.choice(val_species_images)
random_test_image = random.choice(test_species_images)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
images = [random_train_image, random_val_image, random_test_image]
for ax, img_path in zip(axes, images):
    if img_path:
        image = Image.open(img_path)
        ax.imshow(image)
        ax.set_title(f"{img_path.parent.name} - {img_path.name}", fontsize=8, pad=8)
        ax.axis('off')
plt.suptitle('Random Images from Train, Validation, and Test Sets', fontsize=16)
plt.show()

**1.4 Image dimension and aspect ratios**

Image format

In [ ]:
# Analyze image sizes and formats in the dataset

sizes = []
formats = Counter()
image_extensions

for image_file in dataset_path.rglob("*"):
    if image_file.is_file() and image_file.suffix.lower() in image_extensions:
        try:
            with Image.open(image_file) as img:
                sizes.append(img.size)
                formats[img.format] += 1

        except Exception as e:
            print(f"Error processing {image_file}: {e}")
print("Number of images processed:", len(sizes))
print("\nMost common image formats:")

for format, count in formats.most_common(20):
    print(f"Format: {format}, Count: {count}")

for size, count in Counter(sizes).most_common(20):
    print(f"Size: {size}, ->, Count: {count}")

Image dimensiion and aspect ratio

In [ ]:
# Analyze image widths, heights, and aspect ratios in the dataset

widths = []
heights = []
aspect_ratios = []

for image_file in dataset_path.rglob("*"):
    if image_file.is_file() and image_file.suffix.lower() in image_extensions:
        try:
            with Image.open(image_file) as img:
                width, height = img.size
                widths.append(width)
                heights.append(height)
                aspect_ratios.append(width / height)

        except Exception as e:
            print(f"Error processing {image_file}: {e}")

print("Minum width:", min(widths))
print("Maximum width:", max(widths))
print("Minum height:", min(heights))
print("Maximum height:", max(heights))
print("Minum aspect ratio:", min(aspect_ratios))
print("Maximum aspect ratio:", max(aspect_ratios))

print(f"Average width: {sum(widths) / len(widths):.2f}")
print(f"Average height: {sum(heights) / len(heights):.2f}")
print(f"Average aspect ratio: {sum(aspect_ratios) / len(aspect_ratios):.2f}")

Image orientation

In [ ]:
# Count the number of portrait, landscape, and square images in the dataset

portrait = 0
landscape = 0
square = 0
# Count the number of portrait, landscape, and square images
for image_file in dataset_path.rglob("*"):
    if image_file.is_file() and image_file.suffix.lower() in image_extensions:
        try:
            with Image.open(image_file) as img:
                width, height = img.size
                if width > height:
                    landscape += 1
                elif width < height:
                    portrait += 1
                else:
                    square += 1

        except Exception as e:
            print(f"Error processing {image_file}: {e}")

print(f"Portrait images: {portrait}")
print(f"Landscape images: {landscape}")
print(f"Square images: {square}")

**2.5 Image quality and integrity**

Check for potentially low-resolution images

In [ ]:
# Identify low-resolution images in the dataset (width < 244 or height < 224)

low_resolution_images = []
for image_file in dataset_path.rglob("*"):
    if image_file.is_file() and image_file.suffix.lower() in image_extensions:
        try:
            with Image.open(image_file) as img:
                width, height = img.size
                if width < 244 or height < 224:
                    low_resolution_images.append((image_file, (width, height)))

        except Exception as e:
            print(f"Error processing {image_file}: {e}")

print(f"Number of low-resolution images: {len(low_resolution_images)}")
for image, size in low_resolution_images:
    print(f"Image: {image}, Size: {size}")

**2.6 Check image integrity**

PIL integrity check: open every image and report unreadable/corrupt files

In [ ]:
# Check for unreadable or corrupt images in the dataset using PIL's verify() method

bad_images = []

for image_file in dataset_path.rglob("*"):
    if image_file.is_file() and image_file.suffix.lower() in image_extensions:
        try:
            with Image.open(image_file) as img:
                img.verify()
        except Exception as e:
            bad_images.append((str(image_file), str(e)))

print(f"Unreadable/corrupt images: {len(bad_images)}")

if bad_images:
    for path, error in bad_images[:20]:
        print(path, "->", error)
else:
    print("All image files passed the PIL integrity check.")


## 3. Preprocessing decision

The dataset inspection did not identify unreadable images or images with insufficient resolution. Therefore, permanent image cleaning such as deleting, sharpening, or changing image quality was not considered necessary.

The images have different original dimensions and orientations. Instead of modifying the original files, resizing and cropping are performed dynamically during model training using torchvision transforms.

For the training set, random resized cropping and horizontal flipping are used as augmentation. For validation and testing, deterministic preprocessing is used so that these sets are evaluated consistently.

An exact duplicate analysis was also performed. Duplicate images shared between training and validation can introduce redundancy and potentially affect validation reliability. Therefore, the duplicate findings are investigated further after the baseline model and error analysis.

In [ ]:
from torchvision import transforms, datasets
from torchvision.models import resnet50, ResNet50_Weights

# Define the transformations for training and evaluation using ResNet50 weights
weights = ResNet50_Weights.DEFAULT

train_transform = transforms.Compose([
    
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
# Define the evaluation transformations using ResNet50 weights

eval_transform = weights.transforms()

print("Training transform:")
print(train_transform)

print("\nEvaluation transform:")
print(eval_transform)

### Training Data Augmentation
Training images are augmented using random resized cropping and horizontal flipping. A sample image is visualized before and after augmentation to verify that the transformatiosn preserve the important vidual characteristics of the tree



In [ ]:
# Visualize the effect of data augmentation on a sample image from the training set

sample_image_path = sorted(
    file for file in train_path.rglob("*")
    if file.is_file() and file.suffix.lower() in image_extensions
)[0]  

# Load the sample image

original_image = Image.open(sample_image_path).convert("RGB")

augmentation_only = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0), ratio=(0.75, 1.33)),
    transforms.RandomHorizontalFlip(p=0.5)
])

# Visualize the original and augmented images

fig, axes = plt.subplots(2, 3, figsize=(12, 6))
axes[0, 0].imshow(original_image)
axes[0, 0].set_title("Original Image")
axes[0, 0].axis("off")
# Generate and visualize augmented images
for i in range(5):
    augmented_image = augmentation_only(original_image)
    row = (i + 1) // 3
    column = (i + 1) % 3
    axes[row, column].imshow(augmented_image)
    axes[row, column].set_title(f"Augmented Image {i + 1}")
    axes[row, column].axis("off")

plt.suptitle("Data Augmentation Examples", fontsize=16)
plt.tight_layout()
plt.show()

## Data Quality Investigation

Before changing the preprocessing pipeline, an initial RENet50 model was trained on the original dataset. The test errors were than analysed to identify possible sources of classification difficulty.
The analysis focused on:
- incorrect predictions;
- high-confidence incorrect prediction;
- possible class-level confusion; and
- exact duplicate images between the trainign and validation sets.

The findings were then used to determind whether a second trainign round was justified.


In [ ]:
from torchvision.datasets import ImageFolder
# Create datasets for training, validation, and testing using ImageFolder
train_dataset = ImageFolder(root=dataset_path / "train", transform=train_transform)
val_test_transform = eval_transform
# Create datasets for validation and testing using ImageFolder
val_dataset = ImageFolder(
    root=dataset_path / "val",
    transform=val_test_transform
)
test_dataset = ImageFolder(root=dataset_path / "test", transform=val_test_transform)

print(f"Number of training samples: {len(train_dataset)}")
print("\nClasses:")
for i, class_name in enumerate(train_dataset.classes):
    print(f"  {i}: {class_name}")

print(f"Number of validation samples: {len(val_dataset)}")
print(f"Number of test samples: {len(test_dataset)}")

In [ ]:
# Analyze class distribution in the training dataset and calculate the imbalance ratio
train_counts = Counter(
    train_dataset.classes[label] for label in train_dataset.targets
)
# Print the class distribution
largest_class_name = max(train_counts, key=train_counts.get)
smallest_class_name = min(train_counts, key=train_counts.get)

largest_count = train_counts[largest_class_name]
smallest_count = train_counts[smallest_class_name]
imbalance_ratio = largest_count / smallest_count

print(f"Largest class: {largest_class_name} with {largest_count} samples")
print(f"Smallest class: {smallest_class_name} with {smallest_count} samples")
print(f"Imbalance ratio: {imbalance_ratio:.2f}")

**DataLoaders and input verification**

DataLoaders batch the images for model training and evaluation.

* Training uses shuffle=True so batches are randomized each epoch.
* Validation and test use shuffle=False because ordering is not needed for evaluation.

In [ ]:
# Set up DataLoaders for training, validation, and testing

from torch.utils.data import Dataset, DataLoader
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Verify tensor shape, dtype,a and normalization to check one trainign batch before model training.

In [ ]:
# Analyze a batch of images and labels from the training DataLoader

images, labels = next(iter(train_loader))
print(f"Batch of images shape: {images.shape}")
print(f"Batch of labels shape: {labels.shape}")
print("Image dtype:", images.dtype)
print("Label dtype:", labels.dtype)
print("Tensor minimum:", images.min().item())
print("Tensor maximum:", images.max().item())
print("Tensor mean:", images.mean().item())
print("Tensor standard deviation:", images.std().item())

# Part 2: Train model ResNet50

Visualize the transformed images


In [ ]:
# Visualize a batch of normalized training images with their corresponding labels

def visualize_batch(images, labels, class_names, n=8):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    plt.figure(figsize=(16, 8))
    for i in range(min(n, len(images))):
        img = images[i].numpy().transpose((1, 2, 0))
        img = std * img + mean
        img = np.clip(img, 0, 1)
        plt.subplot(2, n // 2, i + 1)
        plt.imshow(img)
        plt.title(class_names[labels[i]])
        plt.axis('off')
    plt.tight_layout()
    plt.show()

# Visualize eight normalized training images from the existing batch
visualize_batch(
    images=images,
    labels=labels,
    class_names=train_dataset.classes,
    n=8
)

Set random seed and device

In [ ]:
# Set random seeds for reproducibility and determine the device (CPU or GPU) to use

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

Define the pretrained weights

In [ ]:
from torchvision import transforms, datasets
from torchvision.models import resnet50, ResNet50_Weights
weights = ResNet50_Weights.DEFAULT
evaluate_transform = weights.transforms()

Initialize ResNet50 

In [ ]:
# Set up the ResNet50 model for transfer learning, freezing the pre-trained layers and modifying the final fully connected layer to match the number of classes in the training dataset
import torch.nn as nn
model = resnet50(weights=weights)

for param in model.parameters():
    param.requires_grad = False 
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, len(train_dataset.classes))
model = model.to(device)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

total_params = sum(p.numel() for p in model.parameters())

print(f"Total parameters in the model: {total_params}")
print(f"Trainable parameters in the model: {trainable_params}")
print(f"Number of tree classes: {len(train_dataset.classes)}")

Define the loss function and optimizer

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001, weight_decay=1e-4)

Train and validate the model

In [ ]:
# Define functions for training and validating the model for one epoch

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()  # Set the model to training mode
    model.fc.train()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)

        running_loss += loss.item() * labels.size(0)
        _, predicted = torch.max(outputs, 1)
        correct_predictions += (predicted == labels).sum().item()
        total_samples += labels.size(0)

    epoch_loss = running_loss / total_samples
    epoch_accuracy = correct_predictions / total_samples

    return epoch_loss, epoch_accuracy

def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            batch_size = labels.size(0)

            running_loss += loss.item() * labels.size(0)
            _, predicted = torch.max(outputs, 1)
            correct_predictions += (predicted == labels).sum().item()
            total_samples += labels.size(0)

    epoch_loss = running_loss / total_samples
    epoch_accuracy = correct_predictions / total_samples

    return epoch_loss, epoch_accuracy

Run the experiment

In [ ]:
import time
import copy
# Train the model for a specified number of epochs, saving the best model based on validation accuracy

def train_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs, model_save_path):
    history = []
    best_val_accuracy = -1.0
    best_epoch = 0
    best_model_state = copy.deepcopy(model.state_dict())
    start_time = time.perf_counter()

    for epoch in range(num_epochs):
        epoch_start_time = time.perf_counter()
        train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_accuracy = validate_one_epoch(model, val_loader, criterion, device)

        epoch_end_time = time.perf_counter()
        epoch_time = epoch_end_time - epoch_start_time

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "val_loss": val_loss,
            "val_accuracy": val_accuracy,
            "epoch_time": epoch_time
        })

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_epoch = epoch + 1
            best_model_state = copy.deepcopy(model.state_dict())

            torch.save(
                {
                    "model_state_dict": best_model_state,
                    "class_names": classes,
                    "class_to_idx": train_dataset.class_to_idx,
                    "best_val_accuracy": best_val_accuracy,
                    "best_epoch": best_epoch
                },
                model_save_path
            )
        print(f"Epoch [{epoch + 1}/{num_epochs}] - "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.4f} - "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.4f} - "
              f"Time: {epoch_time:.2f}s")

    total_time = time.perf_counter() - start_time
    model.load_state_dict(best_model_state)
    print(f"\nTraining completed in {total_time:.2f} seconds.")
    print(f"Best validation accuracy: {best_val_accuracy:.4f} at epoch {best_epoch}")

    return (
        model, 
        history,
        best_val_accuracy,
        best_epoch, total_time
    )

In [ ]:
# Train the model for 10 epochs and save the best model based on validation accuracy

train_original = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    num_epochs=10,
    model_save_path="best_model_original.pth"
)

Display the learning curves

In [ ]:
# Visualize the training history (loss and accuracy) for both training and validation sets

def visualize_training_history(history, model_name):
	
	history_df = pd.DataFrame(history)
	fig, axes = plt.subplots(1, 2, figsize=(12, 6))

	axes[0].plot(history_df['epoch'], history_df['train_accuracy'], label='Train Accuracy', marker='o')
	axes[0].plot(history_df['epoch'], history_df['val_accuracy'], label='Validation Accuracy', marker='o')

	axes[0].set_xlabel('Epoch')
	axes[0].set_ylabel('Accuracy')
	axes[0].set_title(f'Training and Validation Accuracy {model_name}')
	axes[0].legend()

	axes[1].plot(history_df['epoch'], history_df['train_loss'], label='Train Loss', marker='o')
	axes[1].plot(history_df['epoch'], history_df['val_loss'], label='Validation Loss', marker='o')

	axes[1].set_xlabel('Epoch')
	axes[1].set_ylabel('Loss')
	axes[1].set_title(f'Training and Validation Loss {model_name}')
	axes[1].legend()

	plt.tight_layout()
	plt.show()

	return history_df
# Visualize the training history for the original model

history_df = visualize_training_history(
        train_original[1],
        "Original Model"
    )

Generate predictions on the test set

In [ ]:
# Get predictions and probabilities for the test dataset using the trained model

def get_predictions(model, test_loader, device):

    all_labels = []
    all_probs = []
    model.eval()
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            probabilities = torch.softmax(outputs, dim=1)

            all_labels.append(labels)
            all_probs.append(probabilities.cpu())
    all_labels = torch.cat(all_labels).numpy()
    all_probs = torch.cat(all_probs).numpy()

    print(f"All labels shape: {all_labels.shape}")
    print(f"All probabilities shape: {all_probs.shape}")
# Get predictions and probabilities for the test dataset using the trained model
    return all_labels, all_probs

In [ ]:
# Get predictions and probabilities for the test dataset using the trained model

all_labels, all_probs = get_predictions(
    model=train_original[0],
    test_loader=test_loader,
    device=device
)

Accuracy, precision, recall and F1

In [ ]:
# Evaluate the model's performance on the test dataset using various metrics

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
) 
def evaluate_model_performance(all_labels, all_probs):
# Evaluate the model's performance on the test dataset using various metrics
    predicted_labels = np.argmax(all_probs, axis=1)
    accuracy = accuracy_score(all_labels, predicted_labels)
    precision = precision_score(all_labels, predicted_labels, average='macro', zero_division=0)
    recall = recall_score(all_labels, predicted_labels, average='macro', zero_division=0)
    f1 = f1_score(all_labels, predicted_labels, average='macro', zero_division=0)
    cm = confusion_matrix(all_labels, predicted_labels)

    print(f"Test Accuracy: {accuracy:.4f}")
    print(f"Test Precision: {precision:.4f}")
    print(f"Test Recall: {recall:.4f}")
    print(f"Test F1 Score: {f1:.4f}")

    print("\nClassification Report:")
    print(classification_report(all_labels, predicted_labels))

    print("\nConfusion Matrix:")
    print(confusion_matrix(all_labels, predicted_labels))
# Evaluate the model's performance on the test dataset using various metrics
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "confusion_matrix": cm
    }

In [ ]:
# Evaluate the model's performance on the test dataset using various metrics
print("Model Performance on Test Set: Original Model")
original_results = evaluate_model_performance(all_labels, all_probs)

Vizualization confusion matrix

In [ ]:
import seaborn as sns
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=train_dataset.classes, yticklabels=train_dataset.classes)
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('ResNet50 Confusion Matrix')
plt.show()

Calculate top-n accuracy

In [ ]:
# Calculate top-n accuracy for the model's predictions on the test dataset
def top_n_accuracy(y_true, y_probs, n=5):
    top_n_preds = np.argsort(y_probs, axis=1)[:, -n:]
    correct = 0
    for i in range(len(y_true)):
        if y_true[i] in top_n_preds[i]:
            correct += 1
    return correct / len(y_true)
# Calculate top-n accuracy for the model's predictions on the test dataset
num_classes = len(train_dataset.classes)
for n in range(1, min(6, num_classes + 1)):
    acc = top_n_accuracy(all_labels, all_probs, n=n)
    print(f"Top-{n} Accuracy: {acc:.4f}")

Error analysis

Incorrect test predictions were examined to identified patterns that couls explain the model's classification errors.

For each incorrect prediction, the true class, predicted class, prediction confidence, and probalility assigned to the true class were recorded. This maade it possible to distinguish uncertain predictions from case where the model was strongly confident but incorrect.

In [ ]:
# Get prediction from the test set
test_predictions = all_probs.argmax(axis=1)
incorrect_indices = np.where(test_predictions != all_labels)[0]
print(f"Total test samples: {len(all_labels)}")
print(f"Number of incorrect predictions: {len(incorrect_indices)}")
print(f"Correct predictions:{len(all_labels) - len(incorrect_indices)}")
print(f"Test accuracy: {(len(all_labels) - len(incorrect_indices)) / len(all_labels):.4f}")

Create an error-analysis table

In [ ]:
# Create a DataFrame for error analysis, including true labels, predicted labels, confidence scores, and top-5 predictions for incorrect predictions

def create_error_analysis_dataframe(incorrect_indices, all_labels, test_predictions, all_probs, train_dataset):
    error_analysis = []
    for idx in incorrect_indices:
        true_label = all_labels[idx]
        predicted_label = test_predictions[idx]
        predicted_probs = all_probs[idx]
        top_5_indices = np.argsort(predicted_probs)[-5:][::-1]
        top_5_classes = [train_dataset.classes[i] for i in top_5_indices]
        top_5_probs = predicted_probs[top_5_indices]
# Create a DataFrame for error analysis, including true labels, predicted labels, confidence scores, and top-5 predictions for incorrect predictions
        error_analysis.append({
            "index": idx,
            "true_label": train_dataset.classes[true_label],
            "predicted_label": train_dataset.classes[predicted_label],
            "confidence": all_probs[idx][predicted_label],
            "true_class_probability": all_probs[idx][true_label],
            "top_5_classes": top_5_classes,
            "top_5_probabilities": top_5_probs
        })
        # Create a DataFrame for error analysis, including true labels, predicted labels, confidence scores, and top-5 predictions for incorrect predictions
        
    error_analysis_df = pd.DataFrame(error_analysis)
    error_analysis_df = error_analysis_df.sort_values(by="confidence", ascending=False).reset_index(drop=True)
    return error_analysis_df

In [ ]:
# Create a DataFrame for error analysis, including true labels, predicted labels, confidence scores, and top-5 predictions for incorrect predictions
error_analysis_df = create_error_analysis_dataframe(
    incorrect_indices = incorrect_indices,
    all_labels = all_labels,
    test_predictions = test_predictions,
    all_probs = all_probs,
    train_dataset = train_dataset
)

display.display(error_analysis_df.head(20))

Look specifically at high-confidence errors

In [ ]:
high_confidence_errors = error_analysis_df[error_analysis_df["confidence"] > 0.7].copy()
print(f"Number of high-confidence errors (confidence > 0.7): {len(high_confidence_errors)}")
display.display(high_confidence_errors.head(20))

**Exact duplicate analysis**

The analysis identified 10 exact duplicate hash groups between the training and validation sets. Because one hash can correspond to more than one file, the number of duplicate hash groups does not necessarily equal the number of files involved.

**Decision for the second traning round**
Since there're 10 duplicate hash group, a controlled second training round was prepares to investigate the effect of removing these duplicates. 

Tha validation copies were retained and the corresponding train copies were removed while the original dataset was preserved and a seperate cleaned dataset was created.



In [ ]:
# Copy original dataset to new folder for further cleaning and preprocessing

import shutil

cleaned_dataset_path = Path("D:\\NEUTRAL_TOMMI\\GROUPPROJECT\\cleaned_dataset")

if cleaned_dataset_path.exists():
    print(f"Already exists: {cleaned_dataset_path}")
else:
    shutil.copytree(dataset_path, cleaned_dataset_path)
    print(f"Copied dataset to: {cleaned_dataset_path}")

Create function to remove duplicate files

In [ ]:
# Remove duplicate images from the training set based on cross-split duplicates found earlier

def remove_train_duplicates(
        cross_split_duplicates,
        original_dataset_path,
        cleaned_dataset_path,
        dry_run=True):

    files_to_remove = []
    for splits, file_hashes in cross_split_duplicates.items():
        if "train" not in splits or "val" not in splits:
            continue
        for file_hash in file_hashes:
            train_dir = original_dataset_path / "train"
            for train_file in train_dir.rglob("*"):
                if train_file.is_file() and train_file.suffix.lower() in image_extensions:
                    if md5_hash(train_file) == file_hash:
                        realtive_path = train_file.relative_to(original_dataset_path)
                        cleaned_train_file = cleaned_dataset_path / realtive_path
                        files_to_remove.append(cleaned_train_file)
    print(f"Total files to remove from training set: {len(files_to_remove)}")
    
# Using the dry_run parameter to either simulate or actually remove the files from the training set
    if dry_run:
        print("Dry run enabled. No files will be deleted.")
    else:
        print("Removing files from training set...")

    removed_files = []
    for file in files_to_remove:
        if dry_run:
            print(f"Would remove: {file}")
        else:
            if file.exists():
                file.unlink()
                removed_files.append(file)  
                print(f"Removing: {file}")
            else:
                print(f"File not found: {file}")
    if dry_run:
        print(f"File that would be removed: {len(files_to_remove)}")
    else:   
        print(f"Total files removed: {len(removed_files)}")

    return files_to_remove if dry_run else removed_files                  

In [ ]:
print("Original:", dataset_path.resolve())
print("Cleaned:", cleaned_dataset_path.resolve())

print("\nCleaned dataset exists:", cleaned_dataset_path.exists())
# Check if the train, val, and test directories exist in the cleaned dataset
for split in ["train", "val", "test"]:
    split_path = cleaned_dataset_path / split
    print(f"{split} exists:", split_path.exists())

In [ ]:
# Remove duplicate images from the training set based on cross-split duplicates found earlier

file_to_remove = remove_train_duplicates(
    cross_split_duplicates=cross_split_duplicates,
    original_dataset_path=dataset_path,
    cleaned_dataset_path=cleaned_dataset_path,
    dry_run=False
)

Count images in original and cleaned dataset

In [ ]:
original_counts = count_images_in_splits(dataset_path)

cleaned_counts = count_images_in_splits(cleaned_dataset_path)

print("Original dataset:")
for split, count in original_counts.items():
    print(f"  {split}: {count}")

print("\nCleaned dataset:")
for split, count in cleaned_counts.items():
    print(f"  {split}: {count}")

Create the cleaned training dataset

In [ ]:
cleaned_train_dataset = datasets.ImageFolder(
    root=cleaned_dataset_path / "train", 
    transform=train_transform)

cleaned_val_dataset = datasets.ImageFolder(
    root=cleaned_dataset_path / "val",
    transform=eval_transform)

cleaned_test_dataset = datasets.ImageFolder(
    root=cleaned_dataset_path / "test",
    transform=eval_transform)

print(f"Number of training samples: {len(cleaned_train_dataset)}")
print(f"Number of validation samples: {len(cleaned_val_dataset)}")
print(f"Number of test samples: {len(cleaned_test_dataset)}")

Create DataLoaders

In [ ]:
cleaned_train_loader = DataLoader(cleaned_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
cleaned_val_loader = DataLoader(cleaned_val_dataset, batch_size=BATCH_SIZE, shuffle=False)
cleaned_test_loader = DataLoader(cleaned_test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# Set up the ResNet50 model for transfer learning on the cleaned dataset, freezing the pre-trained layers and modifying the final fully connected layer to match the number of classes in the cleaned training dataset
print("Number of classes:", len(cleaned_train_dataset.classes))
print("Model output classes:", cleaned_model.fc.out_features)

print("Trainable parameters:")
for name, param in cleaned_model.named_parameters():
    if param.requires_grad:
        print(name)

In [ ]:
# Set up the optimizer and loss function for training the cleaned model

cleaned_optimizer = torch.optim.Adam(cleaned_model.fc.parameters(), lr=0.001, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

In [ ]:
# Train the cleaned model for 10 epochs and save the best model based on validation accuracy

class_to_idx=cleaned_train_dataset.class_to_idx
print("Class to index mapping:")
for class_name, index in class_to_idx.items():
    print(f"  {class_name}: {index}")
print(cleaned_train_dataset.class_to_idx == train_dataset.class_to_idx)

Initial set up before training

In [ ]:
weights = ResNet50_Weights.DEFAULT
cleaned_model = resnet50(weights=weights)
num_classes = len(cleaned_train_dataset.classes)
cleaned_model.fc = nn.Linear(cleaned_model.fc.in_features, num_classes)
cleaned_model = cleaned_model.to(device)

In [ ]:
# Set up the optimizer and loss function for training the cleaned model

cleaned_model.train()

images, labels = next(iter(cleaned_train_loader))
images = images.to(device)
labels = labels.to(device)

cleaned_optimizer.zero_grad()

outputs = cleaned_model(images)
loss_before = criterion(outputs, labels)

loss_before.backward()
cleaned_optimizer.step()

cleaned_optimizer.zero_grad()

outputs_after = cleaned_model(images)
loss_after = criterion(outputs_after, labels)

print(f"Loss before update: {loss_before.item():.4f}")
print(f"Loss after update:  {loss_after.item():.4f}")

In [ ]:
# Train the cleaned model for 10 epochs and save the best model based on validation accuracy

cleaned_history = train_one_epoch(
    model=cleaned_model,
    dataloader=cleaned_train_loader,
    criterion=criterion,
    optimizer=cleaned_optimizer,
    device=device
)

cleaned_validation = validate_one_epoch(
    model=cleaned_model,
    dataloader=cleaned_val_loader,
    criterion=criterion,
    device=device
)

In [ ]:
# Train the cleaned model for 10 epochs and save the best model based on validation accuracy

train_cleaned_model = train_model(
    model=cleaned_model,
    train_loader=cleaned_train_loader,
    val_loader=cleaned_val_loader,
    criterion=criterion,
    optimizer=cleaned_optimizer,
    device=device,
    num_epochs=10,
    model_save_path="best_model_cleaned.pth"
)

Visualized clean model

In [ ]:
# Visualize the training history for the cleaned model

clean_history_df = visualize_training_history(
        train_cleaned_model[1],
        "ResNet50 - Clean Model"
    )

In [ ]:
# Get predictions and probabilities for the test dataset using the cleaned model

cleaned_all_labels, cleaned_all_probs = get_predictions(
    model=train_cleaned_model[0],
    test_loader=cleaned_test_loader,
    device=device
)

In [ ]:
print("Model Performance on Test Set: Cleaned Model")

cleaned_results = evaluate_model_performance(
    all_labels=cleaned_all_labels,
    all_probs=cleaned_all_probs,
    
)

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=cleaned_test_dataset.classes, yticklabels=cleaned_test_dataset.classes)
plt.xlabel('Predicted Cleaned Labels')
plt.ylabel('True Cleaned Labels')
plt.title('ResNet50 Cleaned Confusion Matrix')
plt.show()

In [ ]:
# Calculate top-n accuracy for the cleaned model's predictions on the test dataset

def top_n_accuracy(y_true, y_probs, n=5):
    top_n_preds = np.argsort(y_probs, axis=1)[:, -n:]
    correct = 0
    for i in range(len(y_true)):
        if y_true[i] in top_n_preds[i]:
            correct += 1
    return correct / len(y_true)

cleaned_num_classes = len(cleaned_test_dataset.classes)
for n in range(1, min(6, cleaned_num_classes + 1)):
    acc = top_n_accuracy(cleaned_all_labels, cleaned_all_probs, n=n)
    print(f"Top-{n} Accuracy: {acc:.4f}")

In [ ]:
# Get prediction from the test set
cleaned_test_predictions = cleaned_all_probs.argmax(axis=1)
incorrect_indices = np.where(cleaned_test_predictions != cleaned_all_labels)[0]
print(f"Total test samples: {len(cleaned_all_labels)}")
print(f"Number of incorrect predictions: {len(incorrect_indices)}")
print(f"Correct predictions:{len(cleaned_all_labels) - len(incorrect_indices)}")
print(f"Test accuracy: {(len(cleaned_all_labels) - len(incorrect_indices)) / len(cleaned_all_labels):.4f}")

In [ ]:
# Create a DataFrame for error analysis, including true labels, predicted labels, confidence scores, and top-5 predictions for incorrect predictions

cleaned_error_analysis = []
for idx in incorrect_indices:
    true_label = cleaned_all_labels[idx]
    predicted_label = cleaned_test_predictions[idx]
    predicted_probs = cleaned_all_probs[idx]
    top_5_indices = np.argsort(predicted_probs)[-5:][::-1]
    top_5_classes = [cleaned_test_dataset.classes[i] for i in top_5_indices]
    top_5_probs = predicted_probs[top_5_indices]

    cleaned_error_analysis.append({
        "index": idx,
        "true_label": cleaned_test_dataset.classes[true_label],
        "predicted_label": cleaned_test_dataset.classes[predicted_label],
        "confidence": cleaned_all_probs[idx][predicted_label],
        "true_class_probability": cleaned_all_probs[idx][true_label],
        "top_5_classes": top_5_classes,
        "top_5_probs": top_5_probs
    })
    # Create a DataFrame for error analysis, including true labels, predicted labels, confidence scores, and top-5 predictions for incorrect predictions
    
error_analysis_df = pd.DataFrame(cleaned_error_analysis)
error_analysis_df = error_analysis_df.sort_values(by="confidence", ascending=False).reset_index(drop=True)
display.display(error_analysis_df.head(20))

In [ ]:
# Filter the error analysis DataFrame to include only high-confidence errors (confidence > 0.8) and display the results

cleaned_high_confidence_errors = error_analysis_df[
	error_analysis_df["confidence"] > 0.8
].copy()

print(
	f"Number of high-confidence errors (confidence > 0.8): "
	f"{len(cleaned_high_confidence_errors)}"
)
display.display(cleaned_high_confidence_errors.head(20))

## Compare 2 training performances

Experiment 1: ResNet50 trained on the original dataset

Experiment 2: ResNet50 trained on the cleaned dataset after removing duplicate training images


In [ ]:

comparison_df = pd.DataFrame({
    "Original Model": [
        len(train_dataset),
        train_original[2],
        len(incorrect_indices),
        len(all_labels) - len(incorrect_indices),
        len(high_confidence_errors),
        original_results["accuracy"],
        original_results["precision"],
        original_results["recall"],
        original_results["f1_score"]
    ],

    "Cleaned Model": [
        len(cleaned_train_dataset),
        train_cleaned_model[2],
        len(cleaned_incorrect_indices),
        len(cleaned_all_labels) - len(cleaned_incorrect_indices),
        len(cleaned_high_confidence_errors),
        cleaned_results["accuracy"],
        cleaned_results["precision"],
        cleaned_results["recall"],
        cleaned_results["f1_score"]
        ]
}, index=[
    "Training Samples",
    "Best Validation Accuracy",
    "Incorrect Predictions",
    "Correct Predictions",
    "High-Confidence Errors",
    "Test Accuracy",
    "Macro Precision",
    "Macro Recall",
    "Macro F1"
])

display.display(comparison_df.round(4))


**Comparision results:**

The ResNet50 models trained on the original and cleaned datasets achieved similar performance across the evaluated metrics. The cleaned dataset contianed fewer duplicate trianing images, while the validation and test sets remianed unchanged. The comparision of accuracy, macro precision, macro recall, and macro F1 shows the differences between 2 experiments. The error analysis were also examined to determin whether duplicate removed affected specific class predictions.